In [2]:
import numpy as np
class SimpleAgent:
    def __init__(self,n_states,n_actions):
        self.q_table=np.zeros((n_states,n_actions))
        self.alpha=0.1
        self.gamma=0.9
        self.epsilon=0.1
    def choose_action(self,state):
        if np.random.rand()<self.epsilon:
            return np.random.choice(len(self.q_table[state]))
        else:
            return np.argmax(self.q_table[state])
    def update(self,state,action,reward,next_state):
        best_next=np.max(self.q_table[next_state])
        self.q_table[state,action]+=self.alpha*(reward+self.gamma*best_next-self.q_table[state,action])
print("---- BASIC AGENT TEST ----")
agent=SimpleAgent(3,2)
agent.update(0,1,10,2)
print("Q-table : \n",agent.q_table)

---- BASIC AGENT TEST ----
Q-table : 
 [[0. 1.]
 [0. 0.]
 [0. 0.]]


In [2]:
states=[0,1,2]
actions=[0,1]
p={
    0: {0:[(1,1.0)], 1:[(2,1.0)]},
    1: {0:[(0,1.0)], 1:[(2,1.0)]},
    2: {0:[(2,1.0)], 1:[(2,1.0)]}

}
r={
    0:{0: 5,1:10},
    1:{0: -1,1:2},
    2:{0: 0,1:0},

}
print("\n===MDP Structure===")
print("states",states)
print("Actions:",actions)
print("Transactions:",p)
print("Rewards:",r)


===MDP Structure===
states [0, 1, 2]
Actions: [0, 1]
Transactions: {0: {0: [(1, 1.0)], 1: [(2, 1.0)]}, 1: {0: [(0, 1.0)], 1: [(2, 1.0)]}, 2: {0: [(2, 1.0)], 1: [(2, 1.0)]}}
Rewards: {0: {0: 5, 1: 10}, 1: {0: -1, 1: 2}, 2: {0: 0, 1: 0}}


In [5]:
import numpy as np
def policy_evaluation(policy,p,r,gamma=0.9,theta=1e-5):
    V=np.zeros(len(policy))
    while True:
        delta=0
        for s in range(len(policy)):
            v=V[s]
            a=policy[s]
            V[s]=r[s][a]+gamma*sum(prob*V[next_state] for next_state, prob in p[s][a])
            delta=max(delta,abs(v-V[s]))
        if delta<theta:
            break
    return V
policy=[0,1,0]
V=policy_evaluation(policy,p,r)
print("=== Policy Evaluation ===")
for i,v in enumerate(V):
    print(f"State {i}: {v:.4f}")

=== Policy Evaluation ===
State 0: 6.8000
State 1: 2.0000
State 2: 0.0000


In [9]:
def policy_improvement(v,p,r,gamma=0.9):
    policy=np.zeros(len(v),dtype=int)
    for s in range(len(v)):
        action_values=[]
        for a in p[s]:
            val=r[s][a]+gamma*sum(
                prob*v[next_state] for next_state,
                prob in p[s][a]
            )
            action_values.append(val)
        policy[s]=np.argmax(action_values)
    return policy
imp_policy=policy_improvement(V,p,r)
print("Improved Policy:",imp_policy)
            

Improved Policy: [1 0 0]


In [15]:
def policy_iteration(p,r,gamma=0.9):
    policy=np.zeros(len(p),dtype=int)
    while True:
        v=policy_evaluation(policy,p,r,gamma)
        new_policy=policy_improvement(v,p,r,gamma)
        if np.array_equal(policy,new_policy):
            break
        policy=new_policy
    return policy,v

opt_policy,opt_v=policy_iteration(p,r)
print("Optimal Policy:",opt_policy)
print("Optimal Value:",opt_v)

Optimal Policy: [0 0 0]
Optimal Value: [21.57891224 18.42102102  0.        ]


In [23]:
def value_iteration(p,r,gamma=0.9,theta=1e-5):
    V=np.zeros(len(p))
    while True:
        delta=0
        for s in range(len(p)):
            v=V[s]
            V[s]=max(
                r[s][a]+gamma*sum(
                    prob*V[next_state] for next_state,
                    prob in p[s][a]
                )
                for a in p[s]
            )
            delta=max(delta,abs(v-V[s]))
        if delta<theta:
            break
        return V
V_opt=value_iteration(p,r)
print("Optimal Value:",V_opt)

Optimal Value: [10.  8.  0.]


In [24]:
import numpy as np
import gymnasium as gym
env=gym.make("FrozenLake-v1")
def monte_carlo(env,episodes=500,gamma=0.9):
    returns={}
    V=np.zeros(env.observation_space.n)
    for ep in range(episodes):
        episode=[]
        state=env.reset()[0] if isinstance(env.reset(),tuple)
        done=False
        while not done:
            action=env.action_space.sample()
            step=env.step(action)
            if len(step)==5:
                next_state,reward,done,_,_=step
            else:
                next_state,reward,done,_=step
            episode.append((state,reward))
            state=next_state
        G=0
        for state,reward in reversed(episode):
            G=gamma*G+reward
            if state not in returns:
                returns[state]=[]
            returns[state].append(G)
            V[state]=np.mean(returns[state])
    return V
print("\n=== MONTE CARLO OUTPUT ===")
V_mc=monte_carlo(env)
for i,v in enumerate(V_mc):
    print(f"State {i}:{v:.4f}")
print("\nGrid View:")
print(V_mc.reshape(4,4))

SyntaxError: expected 'else' after 'if' expression (2950703945.py, line 9)